In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               HistGradientBoostingRegressor)
from sklearn.dummy import DummyRegressor
from sklearn.inspection import permutation_importance


In [16]:
reviews      = pd.read_csv("../data/culture_reviews_claude.csv")
patrimonio   = pd.read_csv("../data/patrimonio.csv")
usuarios     = pd.read_csv("../data/usuarios.csv")
int_usuarios = pd.read_csv("../data/intereses_usuarios.csv")

patrimonio["culture_id"] = patrimonio["Unnamed: 0"] + 1

print(f"Reviews:    {len(reviews):>6}")
print(f"Patrimonio: {len(patrimonio):>6}")
print(f"Usuarios:   {len(usuarios):>6}")

Reviews:      7573
Patrimonio:    577
Usuarios:     2000


In [17]:
AFINIDAD = {
    25: ["Historia"],
    26: ["Ciencias Naturales"],
    27: ["Arte", "Artesanía"],
    28: ["Etnografía"],
    29: ["Patrimonio Cultural", "Religión", "Marítimo", "Otros"],
    11: ["Arte", "Otros"],
}
INTERESES_CULTURALES = [25, 26, 27, 28, 29]

user_interests = (int_usuarios
                  .groupby("id_user")["id_interes"]
                  .apply(set).to_dict())

def calcular_afinidad(uid, tipo_cultura):
    """Score 0-1: cuántos intereses del usuario coinciden con el tipo de lugar."""
    intereses_u = user_interests.get(uid, set())
    score = sum(1 for iid, tipos in AFINIDAD.items()
                if iid in intereses_u and tipo_cultura in tipos)
    return min(score, 3) / 3.0

In [18]:
for iid in INTERESES_CULTURALES:
    usuarios_con = set(int_usuarios[int_usuarios["id_interes"] == iid]["id_user"])
    reviews[f"interes_{iid}"] = reviews["user_id"].apply(lambda u: int(u in usuarios_con))

In [19]:
df = reviews.merge(
    usuarios[["id_user", "age", "sexo", "municipality_id"]],
    left_on="user_id", right_on="id_user", how="left"
).merge(
    patrimonio[[
        "culture_id", "Tipo de Cultura", "Tipo de lugar",
        "valoracion", "num_resenas", "Capacidad", "Tienda",
        "Visita Guiada", "Provincia"
    ]],
    on="culture_id", how="left"
)

In [20]:
df["Tipo de Cultura"] = df["Tipo de Cultura"].fillna("Otros")
df["afinidad"] = df.apply(
    lambda r: calcular_afinidad(r["user_id"], r["Tipo de Cultura"]), axis=1
)

# Imputar valoracion por culture_id (mejor que mediana global)
df["valoracion"] = df.groupby("culture_id")["valoracion"].transform(
    lambda x: x.fillna(x.median())
).fillna(df["valoracion"].median())
df["num_resenas"] = df["num_resenas"].fillna(df["num_resenas"].median())

c:\Users\jlalo\Curso_Data\.venv\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\jlalo\Curso_Data\.venv\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\jlalo\Curso_Data\.venv\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\jlalo\Curso_Data\.venv\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\jlalo\Curso_Data\.venv\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\jlalo\Curso_Data\.venv\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  retur

In [21]:
df = pd.get_dummies(df, columns=["sexo", "Tipo de Cultura", "Tipo de lugar", "Provincia"])

drop_cols = ["user_id", "culture_id", "texto", "created_at", "puntuacion",
             "id_user", "municipality_id"]
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
X = X.select_dtypes(include=["number", "bool"]).copy()
bool_cols = X.select_dtypes(include=["bool"]).columns
X[bool_cols] = X[bool_cols].astype(int)
y = df["puntuacion"]

print(f"Shape X: {X.shape}")
print(f"NaNs:    {X.isna().any().sum()}")
print(f"Features clave: {[c for c in X.columns if any(k in c for k in ['afinidad','interes','valoracion'])]}")

Shape X: (7573, 30)
NaNs:    0
Features clave: ['interes_25', 'interes_26', 'interes_27', 'interes_28', 'interes_29', 'valoracion', 'afinidad']


In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"y media: {y.mean():.3f}  std: {y.std():.3f}")

Train: (6058, 30) | Test: (1515, 30)
y media: 4.023  std: 0.861


In [23]:
models = {
    "Dummy":                DummyRegressor(strategy="mean"),
    "Ridge":                Ridge(alpha=1.0),
    "GradientBoosting":     GradientBoostingRegressor(random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42),
    "RandomForest":         RandomForestRegressor(n_estimators=200,
                                                   random_state=42, n_jobs=-1),
}

results      = []
trained_models = {}

for name, estimator in models.items():
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model",   estimator)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
    mae   = mean_absolute_error(y_test, y_pred)
    r2    = r2_score(y_test, y_pred)
    cv_r2 = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="r2").mean()

    results.append({"modelo": name, "RMSE": round(rmse, 4),
                    "MAE": round(mae, 4), "R2": round(r2, 4), "CV_R2": round(cv_r2, 4)})
    trained_models[name] = pipeline

results_df = (pd.DataFrame(results)
              .sort_values("RMSE")
              .reset_index(drop=True))

best_model_name = results_df.loc[0, "modelo"]
best_model      = trained_models[best_model_name]

print("Comparativa de modelos ordenada por RMSE:")
print(results_df.to_string(index=False))
print(f"\n🏆 Mejor modelo: {best_model_name}")

Comparativa de modelos ordenada por RMSE:
              modelo   RMSE    MAE      R2   CV_R2
    GradientBoosting 0.8101 0.6175  0.1099  0.1391
HistGradientBoosting 0.8175 0.6296  0.0936  0.1058
               Ridge 0.8246 0.6208  0.0776  0.0825
               Dummy 0.8591 0.5798 -0.0010 -0.0013
        RandomForest 0.8761 0.6818 -0.0410 -0.0080

🏆 Mejor modelo: GradientBoosting


In [24]:
perm = permutation_importance(best_model, X_test, y_test,
                               n_repeats=10, random_state=42)
imp_df = (pd.DataFrame({"feature": X.columns,
                         "importance": perm.importances_mean})
          .sort_values("importance", ascending=False)
          .head(15))

print("\nTop 15 features más importantes:")
print(imp_df.to_string(index=False))



Top 15 features más importantes:
                            feature  importance
                         valoracion    0.115249
                        num_resenas    0.060502
                           afinidad    0.019244
                         interes_29    0.010212
                         interes_25    0.002508
                                age    0.001626
                Provincia_Guipúzcoa    0.001296
                          sexo_otro    0.001115
                         interes_27    0.000960
           Tipo de Cultura_Marítimo    0.000894
Tipo de Cultura_Patrimonio Cultural    0.000595
        Tipo de Cultura_Gastronomía    0.000400
                         interes_28    0.000340
          Tipo de Cultura_Artesanía    0.000305
                         interes_26    0.000244


In [25]:
def predecir_puntuacion(user_id, culture_id, pipeline, X_template):
    """
    Predice la puntuación que daría user_id al lugar culture_id.
    X_template: cualquier fila de X para conocer las columnas esperadas.
    """
    fila = X_template.iloc[[0]].copy() * 0  # fila vacía con las columnas correctas

    # Datos del usuario
    u = usuarios[usuarios["id_user"] == user_id].iloc[0]
    fila["age"] = u["age"]
    for iid in INTERESES_CULTURALES:
        usuarios_con = set(int_usuarios[int_usuarios["id_interes"] == iid]["id_user"])
        fila[f"interes_{iid}"] = int(user_id in usuarios_con)
    col_sexo = f"sexo_{u['sexo']}"
    if col_sexo in fila.columns:
        fila[col_sexo] = 1

    # Datos del lugar
    p = patrimonio[patrimonio["culture_id"] == culture_id].iloc[0]
    fila["valoracion"]   = p["valoracion"]
    fila["num_resenas"]  = p["num_resenas"]
    fila["Capacidad"]    = p["Capacidad"]
    fila["Tienda"]       = p["Tienda"]
    fila["Visita Guiada"] = p["Visita Guiada"]
    tipo_cultura = p["Tipo de Cultura"] if pd.notna(p["Tipo de Cultura"]) else "Otros"
    col_tc = f"Tipo de Cultura_{tipo_cultura}"
    if col_tc in fila.columns:
        fila[col_tc] = 1
    col_tl = f"Tipo de lugar_{p['Tipo de lugar']}"
    if col_tl in fila.columns:
        fila[col_tl] = 1
    col_prov = f"Provincia_{p['Provincia']}"
    if col_prov in fila.columns:
        fila[col_prov] = 1

    # Afinidad
    fila["afinidad"] = calcular_afinidad(user_id, tipo_cultura)

    pred = pipeline.predict(fila)[0]
    return round(float(np.clip(pred, 1, 5)), 2)


In [26]:
ejemplo_user    = int(usuarios["id_user"].iloc[0])
ejemplo_culture = int(patrimonio["culture_id"].iloc[0])
pred_ejemplo    = predecir_puntuacion(ejemplo_user, ejemplo_culture, best_model, X)

print(f"Usuario {ejemplo_user} → Lugar {ejemplo_culture}")
print(f"Puntuación predicha: {pred_ejemplo} / 5")

Usuario 1 → Lugar 1
Puntuación predicha: 4.15 / 5
